In [151]:
import kagglehub
import pandas as pd
import os

# 1. Baixa o dataset e descobre o caminho real onde ele foi salvo no seu PC
path = kagglehub.dataset_download("namespaiva/base-varejo")

# 2. Junta o caminho da pasta com o nome do seu arquivo CSV

full_path = os.path.join(path, "Base Varejo.csv")

# 3. Lê o arquivo normalmente usando o Pandas
df = pd.read_csv(full_path, encoding='utf-8', sep=';')

#Criando dicionario para mapear os códigos de estado civil para seus significados de acordo com a descrição do dataset
dic_estado_civil = {1: "Casado/UE", 2: "Divorciado", 3: "Separado", 4: "Solteiro", 5: "Viúvo", 6: "Viúvo(a)"}

print("\nInformações do DataFrame:")
print(df.shape)
print(df.info())
print()
print("Primeiros 5 registros:")
print(df.head())




Informações do DataFrame:
(830000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), str(5)
memory usage: 88.7 MB
None

Primeiros 5 registros:
         DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID     PR_CAT  \
0  

# limpeza

### Limpando e convertendo nomes e tipos de Colunas

- Removendo colunas em branco "Unnamed: 10", "Unnamed: 11", "Unnamed: 12", "Unnamed: 13".
- Renomeando colunas de acordo com a descrição do dataset
- Convetendo data para datetime

In [152]:

# remove as colunas extras que não possuem nome
df = df.drop(columns=["Unnamed: 10", "Unnamed: 11", "Unnamed: 12", "Unnamed: 13"])

# renomeia as colunas para nomes mais amigáveis
df = df.rename(columns={"CO_ID": "N_fiscal", "CL_ID": "ID_Cliente", "CL_GENERO": "Genero", "CL_EC": "Estado_Civil", "CL_FHL": "Qtd_Filhos", "CL_SEG": "Seg_Economica", "PR_ID": "ID_Produto", "PR_CAT": "Cat_Produto",
                         "PR_NOME": "Nome_Produto"})
#converte a coluna "DATA" para o formato de datetime do Pandas
df["DATA"] = pd.to_datetime(
    df["DATA"],
    format="%d/%m/%Y",
    errors="coerce")
#Criando as colunas de Ano, Mês e Dia a partir da coluna "DATA"
df["Ano"] = df["DATA"].dt.year
df["Mes"] = df["DATA"].dt.month
df["Dia"] = df["DATA"].dt.day

## Duplicatas

In [153]:
# Identificando duplicatas
dup_mask = df.duplicated()
print(f"Total de duplicatas: {dup_mask.sum()}")
print()

# # Exemplo das duplicatas encontradas
print("Primeiras 4 duplicatas:")
print(df[dup_mask].head(4).to_string()) 
#removendo as duplicatas
df_limpo = df.drop_duplicates()
print(f"Tamanho original: {len(df)} - Tamanho após remoção de duplicatas: {len(df_limpo)}, foram removidas {(len(df) - len(df_limpo))/len(df)*100:.2f}% de linhas totais que eram duplicadas.")


Total de duplicatas: 96553

Primeiras 4 duplicatas:
         DATA  N_fiscal  ID_Cliente Genero  Estado_Civil  Qtd_Filhos Seg_Economica  ID_Produto Cat_Produto        Nome_Produto   Ano  Mes  Dia
19 2019-02-01      1000         534      M             4           1             C          13   ALIMENTOS              BANANA  2019    2    1
40 2019-02-01      1000         534      M             4           1             C           4   ALIMENTOS             ABACAXI  2019    2    1
46 2019-02-01      1000         534      M             4           1             C          11   ALIMENTOS              AZEITE  2019    2    1
49 2019-02-01      1000         534      M             4           1             C          69     BEBIDAS  REFRIGERANTE LIMaO  2019    2    1
Tamanho original: 830000 - Tamanho após remoção de duplicatas: 733447, foram removidas 11.63% de linhas totais que eram duplicadas.


In [154]:
print(df_limpo.info())
print(df_limpo.head())

<class 'pandas.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   DATA           733447 non-null  datetime64[us]
 1   N_fiscal       733447 non-null  int64         
 2   ID_Cliente     733447 non-null  int64         
 3   Genero         733447 non-null  str           
 4   Estado_Civil   733447 non-null  int64         
 5   Qtd_Filhos     733447 non-null  int64         
 6   Seg_Economica  733447 non-null  str           
 7   ID_Produto     733447 non-null  int64         
 8   Cat_Produto    733447 non-null  str           
 9   Nome_Produto   733447 non-null  str           
 10  Ano            733447 non-null  int32         
 11  Mes            733447 non-null  int32         
 12  Dia            733447 non-null  int32         
dtypes: datetime64[us](1), int32(3), int64(5), str(4)
memory usage: 69.9 MB
None
        DATA  N_fiscal  ID_Cliente Genero  E

### Analisando dados

In [155]:
dic_estado_civil = {1: "Casado/UE", 2: "Divorciado", 3: "Separado", 4: "Solteiro", 5: "Viúvo", 6: "Viúvo(a)"}
print(df_limpo["Estado_Civil"].map(dic_estado_civil).value_counts())

Estado_Civil
Separado      189048
Solteiro      179206
Divorciado    172283
Casado/UE     172010
Viúvo          20900
Name: count, dtype: int64


In [156]:
colunas_categoricas = ["Genero", "Estado_Civil", "Qtd_Filhos", "Seg_Economica", "Cat_Produto"]
for col in colunas_categoricas:
    if col == "Estado_Civil":
        print(f"{(df_limpo[col].map(dic_estado_civil).value_counts()/len(df_limpo[col])*100).round(2)}%")
    else:
        print(f"\nContagem de valores para a coluna '{col}':")
        print(f"{(df_limpo[col].value_counts()/len(df_limpo[col])*100).round(2)}%")
        print("-"*20)



Contagem de valores para a coluna 'Genero':
Genero
F    52.14
M    47.86
Name: count, dtype: float64%
--------------------
Estado_Civil
Separado      25.78
Solteiro      24.43
Divorciado    23.49
Casado/UE     23.45
Viúvo          2.85
Name: count, dtype: float64%

Contagem de valores para a coluna 'Qtd_Filhos':
Qtd_Filhos
0    52.49
2    12.84
3    12.60
1    12.39
4     9.69
Name: count, dtype: float64%
--------------------

Contagem de valores para a coluna 'Seg_Economica':
Seg_Economica
B    63.88
C    27.99
A     8.14
Name: count, dtype: float64%
--------------------

Contagem de valores para a coluna 'Cat_Produto':
Cat_Produto
ALIMENTOS     52.38
HIGIENE       18.77
LIMPEZA       17.54
BEBIDAS        5.22
PET            3.89
ACESSORIOS     1.75
#N/D           0.44
Name: count, dtype: float64%
--------------------


Identificado Valor #N/D nas categorias do produto, verificando se existe algum padrão.

In [157]:
df_analise = df_limpo.query("Cat_Produto == '#N/D'")
print(df_analise.head())
print("-"*20)
print("Os valores de ID_produto com #N/D na coluna 'Cat_Produto' são:")
print(df_analise["ID_Produto"].unique())

          DATA  N_fiscal  ID_Cliente Genero  Estado_Civil  Qtd_Filhos  \
82  2019-02-01      1078         290      F             1           0   
223 2019-02-01      1103         957      M             2           3   
640 2019-02-01      1482         453      F             1           0   
857 2019-02-01      1683         438      F             1           0   
917 2019-02-01      1793         608      F             3           0   

    Seg_Economica  ID_Produto Cat_Produto Nome_Produto   Ano  Mes  Dia  
82              B         107        #N/D         #N/D  2019    2    1  
223             B         107        #N/D         #N/D  2019    2    1  
640             B         107        #N/D         #N/D  2019    2    1  
857             B         107        #N/D         #N/D  2019    2    1  
917             C         107        #N/D         #N/D  2019    2    1  
--------------------
Os valores de ID_produto com #N/D na coluna 'Cat_Produto' são:
[107]


**Decisão**

Já que foi identificado que apenas um produto ficou com o termo #N/D foi decidido trata-lo como desconhecido para manter os demais dados.

In [158]:
df_limpo["Cat_Produto"] = df_limpo["Cat_Produto"].replace("#N/D", "Sem Categoria")
df_limpo["Nome_Produto"] = df_limpo["Nome_Produto"].replace("#N/D", "Sem Nome")

In [159]:
print((df_limpo["Cat_Produto"].value_counts()/len(df["Cat_Produto"])*100).round(2))
print("-"*20)
print((df_limpo["Nome_Produto"].value_counts()/len(df["Nome_Produto"])*100).round(2))

Cat_Produto
ALIMENTOS        46.29
HIGIENE          16.59
LIMPEZA          15.50
BEBIDAS           4.61
PET               3.44
ACESSORIOS        1.55
Sem Categoria     0.39
Name: count, dtype: float64
--------------------
Nome_Produto
PRESUNTO COZIDO          1.53
SARDINHA                 0.80
BANANA                   0.79
ESCOVA DE DENTE          0.79
GEL                      0.79
                         ... 
Sem Nome                 0.39
ABSORVENTE               0.39
ALIMENTO PARA PASSARO    0.39
ALHO                     0.38
ALCOOL                   0.38
Name: count, Length: 118, dtype: float64


In [160]:
df_limpo["Estado_Civil"] = df_limpo["Estado_Civil"].map(dic_estado_civil)
print(df_limpo["Estado_Civil"].value_counts())

Estado_Civil
Separado      189048
Solteiro      179206
Divorciado    172283
Casado/UE     172010
Viúvo          20900
Name: count, dtype: int64


# Analise

In [161]:
#Convertendo estado civil para seus significados usando o dicionário criado anteriormente


colunas_categoricas = ["Genero", "Estado_Civil", "Qtd_Filhos", "Seg_Economica", "Cat_Produto"]
for col in colunas_categoricas:
    print(f"\nContagem de valores para a coluna '{col}':")
    print(f"{(df_limpo[col].value_counts()/len(df_limpo[col])*100).round(2)}%")
    print("-"*20)


Contagem de valores para a coluna 'Genero':
Genero
F    52.14
M    47.86
Name: count, dtype: float64%
--------------------

Contagem de valores para a coluna 'Estado_Civil':
Estado_Civil
Separado      25.78
Solteiro      24.43
Divorciado    23.49
Casado/UE     23.45
Viúvo          2.85
Name: count, dtype: float64%
--------------------

Contagem de valores para a coluna 'Qtd_Filhos':
Qtd_Filhos
0    52.49
2    12.84
3    12.60
1    12.39
4     9.69
Name: count, dtype: float64%
--------------------

Contagem de valores para a coluna 'Seg_Economica':
Seg_Economica
B    63.88
C    27.99
A     8.14
Name: count, dtype: float64%
--------------------

Contagem de valores para a coluna 'Cat_Produto':
Cat_Produto
ALIMENTOS        52.38
HIGIENE          18.77
LIMPEZA          17.54
BEBIDAS           5.22
PET               3.89
ACESSORIOS        1.75
Sem Categoria     0.44
Name: count, dtype: float64%
--------------------


Com base nesses dados podemos ver que a maioria da base é:
- feminina
- é separado
- tem 0 filhos
- é da categoria B
- o segmento de maior venda é o alimenticio


## Analise Quantidade de filhos

### Estatisticas

In [162]:
print("=" * 10 + " Análise estatística da coluna 'Qtd_Filhos' " + "=" * 10)
print(f"Mediana: {df_limpo['Qtd_Filhos'].median()}")
print(f"Média: {df_limpo['Qtd_Filhos'].mean()}")
print(f"Moda: {df_limpo['Qtd_Filhos'].mode().iloc[0] if not df_limpo['Qtd_Filhos'].mode().empty else 'Nenhuma'}")
print(f"Valor Máximo: {df_limpo['Qtd_Filhos'].max()}")
print(f"Valor Mínimo: {df_limpo['Qtd_Filhos'].min()}")
print(f"Desvio Padrão: {df_limpo['Qtd_Filhos'].std()}")
print()
print("Contagem de valores 'Qtd_Filhos':")
contagem_qtd_filhos = pd.DataFrame([df_limpo["Qtd_Filhos"].value_counts(), (df_limpo["Qtd_Filhos"].value_counts()/len(df_limpo["Qtd_Filhos"])*100).round(2)])
# 2. Inverte as linhas e colunas usando o .T
# Também vamos dar nomes claros para as colunas para ficar organizado
contagem_qtd_filhos = contagem_qtd_filhos.T
contagem_qtd_filhos.columns = ['Quantidade', 'Percentual']

# 3. Adiciona o símbolo de % na coluna de Percentual
contagem_qtd_filhos['Percentual'] = contagem_qtd_filhos['Percentual'].map('{:.2f}%'.format)
contagem_qtd_filhos['Quantidade'] = contagem_qtd_filhos['Quantidade'].astype(int)
contagem_qtd_filhos = contagem_qtd_filhos.sort_index(ascending=True)

# Visualizar o resultado
print(contagem_qtd_filhos)


========== Análise estatística da coluna 'Qtd_Filhos' ==========
Mediana: 0.0
Média: 1.1460487260838206
Moda: 0
Valor Máximo: 4
Valor Mínimo: 0
Desvio Padrão: 1.4169173697917072

Contagem de valores 'Qtd_Filhos':
            Quantidade Percentual
Qtd_Filhos                       
0               384986     52.49%
1                90845     12.39%
2                94168     12.84%
3                92407     12.60%
4                71041      9.69%


**Insights**
- Há compradores sem filhos e esses são a maioria (+50%).
- O maior numero de filhos por comprador é 4 e essas são a minoria.

### Agrupamentos

In [163]:
df_limpo.info()

<class 'pandas.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   DATA           733447 non-null  datetime64[us]
 1   N_fiscal       733447 non-null  int64         
 2   ID_Cliente     733447 non-null  int64         
 3   Genero         733447 non-null  str           
 4   Estado_Civil   733447 non-null  str           
 5   Qtd_Filhos     733447 non-null  int64         
 6   Seg_Economica  733447 non-null  str           
 7   ID_Produto     733447 non-null  int64         
 8   Cat_Produto    733447 non-null  str           
 9   Nome_Produto   733447 non-null  str           
 10  Ano            733447 non-null  int32         
 11  Mes            733447 non-null  int32         
 12  Dia            733447 non-null  int32         
dtypes: datetime64[us](1), int32(3), int64(4), str(5)
memory usage: 69.9 MB


In [164]:
qtd_filhos_Estado_Civil = df_limpo.groupby("Qtd_Filhos")["Estado_Civil"].value_counts().rename("Quantidade").reset_index(name="Quantidade")
qtd_filhos_Estado_Civil["Porcentagem"] = qtd_filhos_Estado_Civil["Quantidade"] / len(df_limpo["Estado_Civil"]) * 100
qtd_filhos_Estado_Civil["Porcentagem_Grupo"] = (
    qtd_filhos_Estado_Civil["Quantidade"] / 
    qtd_filhos_Estado_Civil.groupby("Qtd_Filhos")["Quantidade"].transform("sum")
) * 100

# 2. (Opcional) Arredonda o resultado para 2 casas decimais para facilitar a leitura
qtd_filhos_Estado_Civil["Porcentagem_Grupo"] = qtd_filhos_Estado_Civil["Porcentagem_Grupo"].round(2).map('{:.2f}%'.format)
qtd_filhos_Estado_Civil["Porcentagem"] = qtd_filhos_Estado_Civil["Porcentagem"].round(2).map('{:.2f}%'.format)


# Exibe o resultado
print(qtd_filhos_Estado_Civil.to_string(index=False))

 Qtd_Filhos Estado_Civil  Quantidade Porcentagem Porcentagem_Grupo
          0     Solteiro      103546      14.12%            26.90%
          0     Separado      100815      13.75%            26.19%
          0    Casado/UE       95234      12.98%            24.74%
          0   Divorciado       85391      11.64%            22.18%
          1     Solteiro       28906       3.94%            31.82%
          1     Separado       18603       2.54%            20.48%
          1   Divorciado       15733       2.15%            17.32%
          1    Casado/UE       15532       2.12%            17.10%
          1        Viúvo       12071       1.65%            13.29%
          2   Divorciado       26685       3.64%            28.34%
          2     Solteiro       23829       3.25%            25.30%
          2     Separado       23129       3.15%            24.56%
          2    Casado/UE       14063       1.92%            14.93%
          2        Viúvo        6462       0.88%             6

Podemos notar analisando os dados desta perspectiva que:
- Não há solteiros com 4 filhos
- Os solteiros com 0 filhos é o maior grupo da base


In [165]:
#Cria o agrupamento combinando Gênero, Estado Civil e Quantidade de Filhos
agrupamento_completo = df_limpo.groupby(["Genero", "Estado_Civil", "Qtd_Filhos"]).size().reset_index(name="Quantidade")

#Calcula a Porcentagem
agrupamento_completo["Porcentagem"] = (agrupamento_completo["Quantidade"] / len(df_limpo)) * 100
#coloca porcentagem com 2 casas decimais e o símbolo %
agrupamento_completo["Porcentagem"] = agrupamento_completo["Porcentagem"].round(2).map('{:.2f}%'.format)
#Ordena os dados maior para menor por porcentagem
agrupamento_completo = agrupamento_completo.sort_values("Porcentagem", ascending=False)
#Exibe o resultado retirando o índice
print(agrupamento_completo.to_string(index=False))

Genero Estado_Civil  Qtd_Filhos  Quantidade Porcentagem
     F     Solteiro           0       53381       7.28%
     F   Divorciado           0       51693       7.05%
     F     Separado           0       51134       6.97%
     M     Solteiro           0       50165       6.84%
     M     Separado           0       49681       6.77%
     F    Casado/UE           0       47819       6.52%
     M    Casado/UE           0       47415       6.46%
     M   Divorciado           0       33698       4.59%
     M   Divorciado           2       16560       2.26%
     F     Solteiro           1       16305       2.22%
     M    Casado/UE           4       16020       2.18%
     F     Solteiro           3       15307       2.09%
     F     Solteiro           2       15027       2.05%
     F     Separado           2       13970       1.90%
     M   Divorciado           3       13797       1.88%
     M     Separado           4       13718       1.87%
     F     Separado           4       13536     

Podemos notar analisando os dados desta perspectiva que:
- O maior publico é o feminino solteiro com 0 filhos

In [166]:
df_limpo.query("Genero == 'F' and Estado_Civil == 'Solteiro' and Qtd_Filhos == 0")["Cat_Produto"].value_counts()

Cat_Produto
ALIMENTOS        27969
HIGIENE           9924
LIMPEZA           9443
BEBIDAS           2794
PET               2086
ACESSORIOS         932
Sem Categoria      233
Name: count, dtype: int64

Confirmando o dado geral, esse grupo também tem como foco itens alimenticios

Com os agrupamentos e estatisticas podemos ver que a maior parte dos consumidores foca na alimentação e (), quase 

In [167]:
#dados
#Somando quantidade de filhos 0, 1, 2
filhos = df_limpo.query("Qtd_Filhos in [0, 1, 2]").shape[0]/len(df_limpo)*100
filhos = f"{filhos:.2f}%"
print(filhos)
#somando higiene e alimentação
categoria = df_limpo.query("Cat_Produto in ['HIGIENE', 'ALIMENTOS']").shape[0]/len(df_limpo)*100
categoria = f"{categoria:.2f}%"
print(categoria)
#porcentagem segmento economico B
segmento_b = df_limpo.query("Seg_Economica == 'B'").shape[0]/len(df_limpo)*100
segmento_b = f"{segmento_b:.2f}%"
segmento_a = df_limpo.query("Seg_Economica == 'A'").shape[0]/len(df_limpo)*100
segmento_a = f"{segmento_a:.2f}%"
print(segmento_b)
print(segmento_a)
#porcentagem mulheres
mulheres = df_limpo.query("Genero == 'F'").shape[0]/len(df_limpo)*100
mulheres = f"{mulheres:.2f}%"
print(mulheres)

77.72%
71.16%
63.88%
8.14%
52.14%


In [168]:
print("-"*20 +" RELATÓRIO " +"-"*20)
print(f""" Analisando os dados do dataset, podemos verificar que a maioria dos clientes possuem 0, 1 ou 2 filhos, representando {filhos} do total. 
Em relação à categoria de produtos, os clientes que compram produtos de HIGIENE ou ALIMENTOS correspondem a {categoria} do total. 
No segmento econômico, os clientes do segmento A representam {segmento_a} enquanto os do segmento B representam {segmento_b}. 
Além disso, a porcentagem de clientes do sexo feminino é de {mulheres}.
O maior número de clientes é do sexo feminino, solteiros e sem filhos, e a categoria de produto mais comprada por esse grupo é a de ALIMENTOS.
Assim recomenda-se que a empresa foque suas estratégias de marketing e vendas em produtos de ALIMENTOS para o público feminino, solteiro e sem filhos, pois é o grupo mais representativo dentro do dataset.
Ou pode focar em tentar atrair clientes do segmento A, que representam uma porcentagem significativa do total, oferecendo produtos de alta qualidade e serviços exclusivos para esse público.
""")

-------------------- RELATÓRIO --------------------
 Analisando os dados do dataset, podemos verificar que a maioria dos clientes possuem 0, 1 ou 2 filhos, representando 77.72% do total. 
Em relação à categoria de produtos, os clientes que compram produtos de HIGIENE ou ALIMENTOS correspondem a 71.16% do total. 
No segmento econômico, os clientes do segmento A representam 8.14% enquanto os do segmento B representam 63.88%. 
Além disso, a porcentagem de clientes do sexo feminino é de 52.14%.
O maior número de clientes é do sexo feminino, solteiros e sem filhos, e a categoria de produto mais comprada por esse grupo é a de ALIMENTOS.
Assim recomenda-se que a empresa foque suas estratégias de marketing e vendas em produtos de ALIMENTOS para o público feminino, solteiro e sem filhos, pois é o grupo mais representativo dentro do dataset.
Ou pode focar em tentar atrair clientes do segmento A, que representam uma porcentagem significativa do total, oferecendo produtos de alta qualidade e servi

In [ ]:
#gerando excel da base limpa
df_limpo.to_excel("Base_Varejo_Limpa.xlsx", index=False)